## Customer Churn Analytics Platform

### Notebook 02 - Data Validation

#### Objective
Validate the Bronze dataset before moving to the Silver layer.

#### Output
Data Quality Report


#### Imports

In [ ]:
# Local Jupyter / GitHub compatibility (no-op on Databricks)
import os
import sys
from pathlib import Path

if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    _root = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "utils" / "databricks_compat.py").exists()
        ),
        None,
    )
    if _root and str(_root / "src") not in sys.path:
        sys.path.insert(0, str(_root / "src"))
    from utils.databricks_compat import inject_globals

    inject_globals(globals())


In [ ]:

# ==========================================================
# Project Initialization
# ==========================================================

import sys

PROJECT_ROOT = "/Workspace/Users/thakurshivek777@gmail.com/Customer-Churn-Platform"
SRC_PATH = f"{PROJECT_ROOT}/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print("Project initialized successfully.")

In [ ]:

from pyspark.sql import functions as F

from config.settings import BRONZE_TABLE

from validation.data_validator import (
    get_row_count,
    get_column_count,
    duplicate_count,
    validate_primary_key,
    blank_value_count,
    missing_value_report,
)

In [ ]:

bronze_df = spark.read.table(BRONZE_TABLE)

display(bronze_df.limit(5))


#### Structural Validation

In [ ]:

print("=" * 70)
print("STRUCTURAL VALIDATION")
print("=" * 70)

rows = get_row_count(bronze_df)
columns = get_column_count(bronze_df)
duplicates = duplicate_count(bronze_df)

print(f"Rows       : {rows}")
print(f"Columns    : {columns}")
print(f"Duplicates : {duplicates}")


#### Business Rule Validation

In [ ]:

primary_key_valid = validate_primary_key(
    bronze_df,
    "customerID"
)

print(f"Primary Key Valid : {primary_key_valid}")

In [ ]:

blank_totalcharges = blank_value_count(
    bronze_df,
    "TotalCharges"
)

print(
    f"Blank TotalCharges : {blank_totalcharges}"
)

In [ ]:

print("Distinct Churn Values")

bronze_df.select("Churn").distinct().show()

In [ ]:

print("=" * 70)
print("MISSING VALUE REPORT")
print("=" * 70)

missing_value_report(bronze_df).show()

In [ ]:

validation_report = {
    "Rows": rows,
    "Columns": columns,
    "Duplicate Rows": duplicates,
    "Primary Key Valid": primary_key_valid,
    "Blank TotalCharges": blank_totalcharges,
}

In [ ]:

print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

for key, value in validation_report.items():
    print(f"{key:<25}: {value}")